# C-MAPSS RUL — Documentation & Analysis

Sensor importance, comparison plots, epoch metrics, and verified report generation.

In [ ]:
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RESULTS = ROOT / 'results'
DOCS = ROOT / 'docs'
MODELS = ROOT / 'models'
for p in (RESULTS, DOCS, MODELS):
    p.mkdir(exist_ok=True)

DATA_DIR = ROOT / 'data'
for candidate in [ROOT / 'data', ROOT, ROOT.parent,
                  Path(r'C:\kaggle\input\cmapss-jet-engine-simulated-data'),
                  Path('/kaggle/input/cmapss-jet-engine-simulated-data')]:
    if candidate.is_dir() and list(candidate.glob('train_FD*.txt')):
        DATA_DIR = candidate
        break
print('ROOT:', ROOT)
print('DATA_DIR:', DATA_DIR)
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

In [ ]:
# Re-use ML helpers (run notebook 01 first, or define here)
import os, glob, warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings('ignore')
RUL_CAP, ROLLING_WINDOW = 125, 5
DATASET_IDS = ['FD001','FD002','FD003','FD004']
COL_NAMES = ['unit','cycle','op_setting_1','op_setting_2','op_setting_3'] + [f'sensor_{i+1}' for i in range(21)]

def discover_files(data_dir):
    mapping = {}
    for path in glob.glob(os.path.join(str(data_dir), '*')):
        name = os.path.basename(path).upper()
        for fd in DATASET_IDS:
            if fd in name:
                mapping.setdefault(fd, {})
                if 'TRAIN' in name: mapping[fd]['train'] = path
                if 'TEST' in name: mapping[fd]['test'] = path
                if 'RUL' in name: mapping[fd]['rul'] = path
    return mapping

def load_dataset(train_path, test_path, rul_path):
    train = pd.read_csv(train_path, sep=r'\s+', header=None, names=COL_NAMES)
    train['RUL'] = train.groupby('unit')['cycle'].transform('max') - train['cycle']
    test = pd.read_csv(test_path, sep=r'\s+', header=None, names=COL_NAMES)
    rul = pd.read_csv(rul_path, sep=r'\s+', header=None, names=['RUL_final'])
    max_cycle = test.groupby('unit')['cycle'].max().reset_index().sort_values('unit')
    max_cycle['RUL_final'] = rul['RUL_final'].values
    test = test.merge(max_cycle[['unit','RUL_final']], on='unit', how='left')
    test['RUL'] = test.groupby('unit')['cycle'].transform('max') - test['cycle'] + test['RUL_final']
    return train, test.drop(columns=['RUL_final'])

def select_features(train_df, cols, min_std=1e-6):
    stds = train_df[cols].std()
    return stds[stds > min_std].index.tolist()

def build_feature_matrix(df, base_cols, window=5):
    parts, grouped = [], df.groupby('unit', sort=False)
    for col in base_cols:
        parts += [df[[col]],
            grouped[col].transform(lambda s: s-s.iloc[0]).to_frame(f'{col}_rel'),
            grouped[col].transform(lambda s: s.rolling(window,min_periods=1).mean()).to_frame(f'{col}_rmean'),
            grouped[col].transform(lambda s: s.rolling(window,min_periods=1).std().fillna(0)).to_frame(f'{col}_rstd')]
    return pd.concat(parts, axis=1)

def prepare_xy(train_df, test_df, feature_cols):
    train_feat = build_feature_matrix(train_df, feature_cols)
    test_feat = build_feature_matrix(test_df, feature_cols)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_feat)
    X_test = scaler.transform(test_feat)
    return X_train, train_df['RUL'].clip(upper=RUL_CAP).values, X_test, test_df['RUL'].values, list(train_feat.columns)

file_map = discover_files(DATA_DIR)

## Load all benchmark results

In [ ]:
ml = pd.read_csv(RESULTS / 'all_model_results.csv'); ml['Family'] = 'ML'
dl = pd.read_csv(RESULTS / 'dl_model_results.csv'); dl['Family'] = 'DL'
g = pd.read_csv(RESULTS / 'graph_model_results.csv'); g['Family'] = 'Graph'
all_df = pd.concat([
    ml.rename(columns={'RMSE_last':'RMSE','MAE_last':'MAE','R2_last':'R2'}),
    dl.rename(columns={'RMSE_last':'RMSE','MAE_last':'MAE','R2_last':'R2'}),
    g.rename(columns={'RMSE_last':'RMSE','MAE_last':'MAE','R2_last':'R2'}),
], ignore_index=True)[['Model','Dataset','Family','RMSE','MAE','R2']]
all_df = all_df.drop_duplicates(subset=['Model','Dataset'], keep='last')
all_df.to_csv(RESULTS / 'all_results_master.csv', index=False)
best = all_df.loc[all_df.groupby('Dataset')['RMSE'].idxmin()]
best.to_csv(RESULTS / 'best_per_dataset_all.csv', index=False)
display(best.round(3))

## ML epoch metrics (stages 10 / 50 / 100 / 200)

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
import xgboost as xgb

CHECKPOINTS = [10, 50, 100, 200]
epoch_rows = []
for fd in DATASET_IDS:
    train_df, test_df = load_dataset(file_map[fd]['train'], file_map[fd]['test'], file_map[fd]['rul'])
    candidate = [c for c in train_df.columns if c.startswith('sensor_') or c.startswith('op_setting_')]
    feature_cols = select_features(train_df, candidate)
    X_train, y_train, X_test, y_test, _ = prepare_xy(train_df, test_df, feature_cols)
    last_idx = test_df.groupby('unit')['cycle'].idxmax().values
    for stage in CHECKPOINTS:
        for name, model in [
            ('XGBoost', xgb.XGBRegressor(n_estimators=stage, max_depth=8, learning_rate=0.05, random_state=42, n_jobs=-1)),
            ('ExtraTrees', ExtraTreesRegressor(n_estimators=stage, max_depth=16, n_jobs=-1, random_state=42)),
            ('HistGradientBoosting', HistGradientBoostingRegressor(max_iter=stage, learning_rate=0.06, max_depth=12, random_state=42)),
        ]:
            model.fit(X_train, y_train)
            pred = model.predict(X_test)
            rmse = float(np.sqrt(mean_squared_error(y_test[last_idx], pred[last_idx])))
            epoch_rows.append({'Model': name, 'Dataset': fd, 'Epoch': stage, 'RMSE': rmse})
epoch_ml = pd.DataFrame(epoch_rows)
epoch_ml.to_csv(DOCS / 'epoch_metrics_ml.csv', index=False)
display(epoch_ml.pivot_table(index='Model', columns='Epoch', values='RMSE').round(2))

## Sensor importance (ExtraTrees)

In [ ]:
FIGURES = DOCS / 'figures'
FIGURES.mkdir(exist_ok=True)
imp_rows = []
for fd in ['FD001', 'FD002', 'FD003', 'FD004']:
    train_df, test_df = load_dataset(file_map[fd]['train'], file_map[fd]['test'], file_map[fd]['rul'])
    candidate = [c for c in train_df.columns if c.startswith('sensor_') or c.startswith('op_setting_')]
    feature_cols = select_features(train_df, candidate)
    X_train, y_train, _, _, expanded = prepare_xy(train_df, test_df, feature_cols)
    m = ExtraTreesRegressor(n_estimators=200, max_depth=24, n_jobs=-1, random_state=42)
    m.fit(X_train, y_train)
    for feat, val in zip(expanded, m.feature_importances_):
        key = feat.split('_r')[0] if '_r' in feat else feat
        for sfx in ['_rel','_rmean','_rstd']:
            if key.endswith(sfx): key = key[:-len(sfx)]
        imp_rows.append({'Dataset': fd, 'Feature': key, 'Importance': float(val)})
sensor_imp = pd.DataFrame(imp_rows).groupby(['Dataset','Feature'])['Importance'].sum().reset_index()
sensor_imp.to_csv(DOCS / 'sensor_importance.csv', index=False)
for fd in ['FD001','FD004']:
    sub = sensor_imp[sensor_imp.Dataset==fd].nlargest(10, 'Importance')
    sub.plot.barh(x='Feature', y='Importance', figsize=(8,5), title=f'Top features {fd}', legend=False)
    plt.tight_layout()
    plt.savefig(FIGURES / f'sensor_importance_{fd}.png', dpi=150)
    plt.show()

## Comparison plots

In [ ]:
pivot = all_df.pivot_table(index='Model', columns='Dataset', values='RMSE')
plt.figure(figsize=(8, max(5, len(pivot)*0.35)))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn_r')
plt.title('RMSE heatmap (last-cycle)')
plt.tight_layout()
plt.savefig(FIGURES / 'heatmap_rmse_all.png', dpi=150)
plt.show()
for fd in DATASET_IDS:
    sub = all_df[all_df.Dataset==fd].sort_values('RMSE')
    plt.figure(figsize=(9, max(4, len(sub)*0.35)))
    sns.barplot(data=sub, y='Model', x='RMSE', hue='Family', dodge=False)
    plt.title(f'{fd} — RMSE last-cycle')
    plt.tight_layout()
    plt.savefig(FIGURES / f'compare_{fd}_RMSE_last.png', dpi=150)
    plt.show()

## View pre-computed DL epoch metrics & report

In [ ]:
dl_epochs = DOCS / 'epoch_metrics_dl.csv'
if dl_epochs.exists():
    display(pd.read_csv(dl_epochs).pivot_table(index='Model', columns='Epoch', values='RMSE').round(2))
report = DOCS / 'CMAPSS_RUL_Benchmark_Documentation.md'
if report.exists():
    from IPython.display import Markdown
    display(Markdown(report.read_text(encoding='utf-8')[:4000] + '\n\n...'))
else:
    print('Run verify cell below to generate report from CSVs.')

## Regenerate verified markdown report

In [ ]:
def df_to_md(df):
    cols = list(df.columns)
    lines = ['| ' + ' | '.join(str(c) for c in cols) + ' |', '| ' + ' | '.join('---' for _ in cols) + ' |']
    for _, row in df.iterrows():
        lines.append('| ' + ' | '.join(str(row[c]) for c in cols) + ' |')
    return '\n'.join(lines)

b = {r.Dataset: r for _, r in best.iterrows()}
lines = ['# NASA C-MAPSS RUL — Benchmark Documentation\n',
         '*Metrics verified from results/*.csv*\n',
         '## Best Model Per Dataset\n']
bt = best[['Dataset','Model','Family','RMSE','MAE','R2']].copy().round(4)
lines.append(df_to_md(bt))
lines.append('\n## All Models RMSE\n')
lines.append(df_to_md(all_df.pivot_table(index='Model', columns='Dataset', values='RMSE').round(2).reset_index()))
(DOCS / 'CMAPSS_RUL_Benchmark_Documentation.md').write_text('\n'.join(lines), encoding='utf-8')
print('Report saved:', DOCS / 'CMAPSS_RUL_Benchmark_Documentation.md')